In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import cfgrib

In [2]:
ds = cfgrib.open_datasets('ERA5/adaptor.mars.internal-1717355356.2162263-17481-13-2d67889f-d258-46d8-8806-001b74a90fbb.grib')

In [3]:
ds_0 = ds[0].to_array().to_pandas().T
# select value at lon -64.8 due to land-sea mask parameter
ds_1 = ds[1].sel(longitude=-64.8).to_array().to_pandas().T
ds_2 = ds[2].sel(longitude=-64.8).to_array().to_pandas().T

In [4]:
ds

[<xarray.Dataset> Size: 20kB
 Dimensions:     (time: 1012)
 Coordinates:
     number      int64 8B 0
   * time        (time) datetime64[ns] 8kB 1940-01-01 1940-02-01 ... 2024-04-01
     step        timedelta64[ns] 8B 00:00:00
     meanSea     float64 8B 0.0
     latitude    float64 8B 10.4
     longitude   float64 8B -64.8
     valid_time  (time) datetime64[ns] 8kB ...
 Data variables:
     tauoc       (time) float32 4kB 0.9436 0.9599 0.9549 ... 0.9592 0.9568 0.9555
 Attributes:
     GRIB_edition:            1
     GRIB_centre:             ecmf
     GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
     GRIB_subCentre:          0
     Conventions:             CF-1.7
     institution:             European Centre for Medium-Range Weather Forecasts,
 <xarray.Dataset> Size: 65kB
 Dimensions:     (time: 1012, longitude: 2)
 Coordinates:
     number      int64 8B 0
   * time        (time) datetime64[ns] 8kB 1940-01-01 1940-02-01 ... 2024-04-01
     step        timed

In [5]:
ds_1.index = ds_0.index
ds_2.index = ds_0.index

In [6]:
ds_merged = pd.concat([ds_0,ds_1,ds_2], axis=1)
ds_merged_2 = ds_merged.reset_index()

In [7]:
ds_merged_2

variable,time,tauoc,sst,sp,u10,v10,lsm,si10,ewss,e,ro,tp,mtpr
0,1940-01-01,0.943567,299.179443,101163.968750,-5.897113,-4.075964,0.066636,7.412658,-7883.234375,-0.004005,0.000000e+00,0.000063,7.451425e-07
1,1940-02-01,0.959873,297.816406,101103.296875,-5.311798,-4.397612,0.066636,7.282772,-7164.238281,-0.002397,0.000000e+00,0.000010,1.132374e-07
2,1940-03-01,0.954855,298.447510,101012.437500,-5.305734,-4.637102,0.066636,7.445036,-7379.441406,-0.002945,0.000000e+00,0.000073,8.581982e-07
3,1940-04-01,0.958487,298.733398,101064.718750,-4.506469,-3.613885,0.066636,6.151993,-5068.019531,-0.002066,1.144586e-06,0.000809,9.358084e-06
4,1940-05-01,0.982341,299.454102,101025.265625,-3.318252,-2.545523,0.066636,4.964498,-3335.750000,-0.001895,3.052231e-06,0.001015,1.174849e-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1007,2023-12-01,0.953388,299.763428,101036.906250,-4.752415,-3.660954,0.066636,6.216632,-4769.929688,-0.002146,7.630579e-07,0.000164,1.895588e-06
1008,2024-01-01,0.946858,299.245850,101241.187500,-5.574019,-4.132046,0.066636,7.249585,-7113.636719,-0.002809,0.000000e+00,0.000050,5.662667e-07
1009,2024-02-01,0.959159,299.254150,101139.906250,-4.173535,-3.709495,0.066636,6.486409,-5325.242188,-0.002234,0.000000e+00,0.000500,5.796259e-06
1010,2024-03-01,0.956806,298.339111,101100.015625,-4.571066,-4.325170,0.066636,6.722514,-5554.187500,-0.001523,0.000000e+00,0.000011,1.221725e-07


In [9]:
# source: https://stackoverflow.com/a/62947565

from rpy2 import robjects
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

# Convert pandas dataframe to R dataframe
with localconverter(robjects.default_converter + pandas2ri.converter):
    r_df = robjects.conversion.py2rpy(ds_merged_2)

# Save R dataframe as .rds file
r_file = "processed/ERA5_data.RDS"
robjects.r.assign("ERA5", r_df)
robjects.r(f"saveRDS(ERA5, '{r_file}')")

<rpy2.rinterface_lib.sexp.NULLType object at 0x1301fd290> [0]